In [1]:
import ee
import geemap
import warnings
import os
warnings.filterwarnings('ignore')
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point, Polygon
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [2]:
study_area = gpd.read_file('Ga_Mantse.geojson')

In [3]:
geemap.ee_initialize()

m = geemap.Map()

Enter verification code:  4/1Aci98E-JDZf6TnXokLtM-BNsjDOGYurwh4qn2XcEEmiUXSUs82uk6dKW6uU



Successfully saved authorization token.


In [4]:
geojson_path  = 'Ga_Mantse.geojson'
roi = geemap.geojson_to_ee(geojson_path)

In [5]:
# Define the study period
start_date = '2025-01-01'
end_date = '2025-12-31'

In [6]:
# Create a list of day offsets (0, 1, 2, ... 364)
n_days = ee.Date(end_date).difference(ee.Date(start_date), 'day')
day_list = ee.List.sequence(0, n_days.subtract(1))

In [7]:
# 1. Collection
era5_land = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")

def extract_weather(day_offset):
    start = ee.Date(start_date).advance(day_offset, 'day')
    end = start.advance(1, 'day')
    
    # Filter collection for the day
    daily_data = era5_land.filterDate(start, end)
    
    # 1. Temperature in Celsius
    temp_img = daily_data.select('temperature_2m').mean().subtract(273.15)
    
    # 2. Dewpoint in Celsius
    dew_img = daily_data.select('dewpoint_temperature_2m').mean().subtract(273.15)
    
    # 3. Calculate Relative Humidity (RH) using Magnus Formula
    # RH = 100 * (exp((17.625 * TD) / (243.04 + TD)) / exp((17.625 * T) / (243.04 + T)))
    def calc_rh(t, td):
        v_air = td.multiply(17.625).divide(td.add(243.04)).exp()
        v_sat = t.multiply(17.625).divide(t.add(243.04)).exp()
        return v_air.divide(v_sat).multiply(100).rename('rh_percent')

    rh_img = calc_rh(temp_img, dew_img)
    
    # Combine bands and downscale/resample to 1km
    combined = temp_img.rename('temp_c') \
        .addBands(rh_img) \
        .resample('bilinear')
    
    # Sample every 1km
    points = combined.sampleRegions(
        collection=roi, 
        scale=1000, 
        geometries=True
    )
    
    return points.map(lambda f: f.set('date', start.format('YYYY-MM-DD')))

# Export Task
weather_fc = ee.FeatureCollection(day_list.map(extract_weather)).flatten()

In [10]:
task_temp = ee.batch.Export.table.toDrive(
    collection=weather_fc,
    description='Ghana_rh_TimeSeries',
    fileFormat='CSV'
)
task_temp.start()

In [11]:
rh = pd.read_csv('Ghana_rh_TimeSeries.csv')

In [12]:
rh.head()

,system:index,CONSTITUEN,ID,date,rh_percent,temp_c,.geo
0,0_0_0,OKAIKWEI NORTH,125,2025-01-01,73.407407,28.685861,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,0_0_1,OKAIKWEI NORTH,125,2025-01-01,73.384360,28.692106,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,0_0_2,OKAIKWEI NORTH,125,2025-01-01,73.361313,28.698352,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,0_0_3,OKAIKWEI NORTH,125,2025-01-01,73.303652,28.689568,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,0_0_4,OKAIKWEI NORTH,125,2025-01-01,73.281112,28.695813,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [13]:
# 2. Function to extract coordinates from the .geo string
def unpack_geo(geo_str):
    # Parse the JSON string
    geo_dict = json.loads(geo_str)
    # Get the coordinates list [lon, lat]
    lon, lat = geo_dict['coordinates']
    return Point(lon, lat)

In [14]:
df = rh.copy()

In [15]:
# 3. Apply the function to create a geometry column
df['geometry'] = df['.geo'].apply(unpack_geo)

# 4. Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# 5. Clean up: Drop the messy .geo and system:index columns
gdf = gdf.drop(columns=['.geo', 'system:index'])

# 6. Optional: Create separate Lon/Lat columns for your ML model
gdf['longitude'] = gdf.geometry.x
gdf['latitude'] = gdf.geometry.y
gdf.head()

,CONSTITUEN,ID,date,rh_percent,temp_c,geometry,longitude,latitude
0,OKAIKWEI NORTH,125,2025-01-01,73.407407,28.685861,POINT (-0.24704 5.60100),-0.247037,5.600996
1,OKAIKWEI NORTH,125,2025-01-01,73.384360,28.692106,POINT (-0.23805 5.60100),-0.238054,5.600996
2,OKAIKWEI NORTH,125,2025-01-01,73.361313,28.698352,POINT (-0.22907 5.60100),-0.229070,5.600996
3,OKAIKWEI NORTH,125,2025-01-01,73.303652,28.689568,POINT (-0.24704 5.60998),-0.247037,5.609979
4,OKAIKWEI NORTH,125,2025-01-01,73.281112,28.695813,POINT (-0.23805 5.60998),-0.238054,5.609979


In [16]:
ksi2 = gdf.to_crs(epsg=4326)

# 5. Save to GeoJSON
ksi2.to_file("Garh_study_area.geojson", driver='GeoJSON')